Importing The Required Libraries

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
import joblib

Loading the data path

In [28]:
df = pd.read_csv('/content/Mall_Customers.csv')

In [29]:
# Create spending categories (our target variable)
bins = [0, 40, 70, 100]
labels = ['Low', 'Medium', 'High']
df['Spending_Category'] = pd.cut(df['Spending Score (1-100)'], bins=bins, labels=labels)

# Drop the original spending score column
df = df.drop('Spending Score (1-100)', axis=1)

# Check class distribution
print(df['Spending_Category'].value_counts())

Spending_Category
Medium    83
Low       63
High      54
Name: count, dtype: int64


Feature Engineering and Preprocessing

In [30]:
# Define features and target
X = df.drop(['CustomerID', 'Spending_Category'], axis=1)
y = df['Spending_Category']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing pipeline
numeric_features = ['Age', 'Annual Income (k$)']
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_features = ['Genre']
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Check the transformed features
preprocessor.fit(X_train)
print("Feature names:", preprocessor.get_feature_names_out())

Feature names: ['num__Age' 'num__Annual Income (k$)' 'cat__Genre_Female'
 'cat__Genre_Male']


Model Building and Evaluation

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

Evaluating the model

In [32]:
# Train and evaluate each model
results = {}
for name, model in models.items():
    # Create pipeline
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Train model
    clf.fit(X_train, y_train)

    # Make predictions
    y_pred = clf.predict(X_test)

    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    results[name] = {
        'model': clf,
        'accuracy': accuracy,
        'report': report,
        'confusion_matrix': cm
    }

    print(f"\n{name} Results:")
    print(f"Accuracy: {accuracy:.2f}")
    print("Classification Report:")
    print(report)


Logistic Regression Results:
Accuracy: 0.47
Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.36      0.36        11
           1       0.40      0.17      0.24        12
           2       0.54      0.76      0.63        17

    accuracy                           0.47        40
   macro avg       0.44      0.43      0.41        40
weighted avg       0.45      0.47      0.44        40


Decision Tree Results:
Accuracy: 0.85
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.73      0.80        11
           1       0.82      0.75      0.78        12
           2       0.85      1.00      0.92        17

    accuracy                           0.85        40
   macro avg       0.85      0.83      0.83        40
weighted avg       0.85      0.85      0.85        40


Random Forest Results:
Accuracy: 0.82
Classification Report:
              precision    recall  f1-score   

Making Predictions on New Data

In [33]:
# Access the decision tree model
decision_tree_model = results['Decision Tree']['model']

# Example of new data (replace with your actual data)
new_data = pd.DataFrame({
    'Genre': ['Male'],
    'Age': [30],
    'Annual Income (k$)': [60]
})

# Make predictions
new_predictions = decision_tree_model.predict(new_data)

# Inverse transform the prediction to get original labels
new_predictions_labels = label_encoder.inverse_transform(new_predictions)
print(f"Prediction for new data: {new_predictions_labels}")

Prediction for new data: ['Medium']


Storing The Model

In [34]:
# Save the Decision Tree model to a file
joblib.dump(decision_tree_model, 'decision_tree_model.joblib')
print("Decision Tree model saved to decision_tree_model.joblib")

Decision Tree model saved to decision_tree_model.joblib
